# Entrenamiento y evaluación de modelos para predecir la variable **default** 

In [244]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path().resolve().parent.parent / "data/data-09-2025"
data_file = "cleaned_data_default.parquet"

df = pd.read_parquet(DATA_DIR / data_file)
df.head()

,plazo,vinculacion,v_cuota,v_prestamo,s_capital,s_intereses,aportes,garantias,valorgarantia,ctasahorros,...,actividadeconomica,estado_cliente,departamento,sexo,curtotalingresos,curtotalegresos,intestrato,actualizacion,default,puntaje_data
n_credito,,,,,,,,,,,,,,,,,,,,,
003-002-0125852-7,1827,8103,356849.0,15000000.0,12923538.0,123855,7741255,1,7741255,33042953.0,...,asalariados,1,antioquia,0,4597000.0,1500000.0,5.0,1,0,795.0
004-002-0068475-5,1826,1434,2650409.0,100460000.0,31911361.0,263265,4601706,1,4601706,3791115.0,...,asalariados,1,antioquia,0,4597000.0,650000.0,5.0,1,0,836.0
003-002-0122592-9,1826,573,791482.0,30000000.0,23844684.0,261477,530431,1,530431,94435.0,...,asalariados,1,antioquia,0,4400000.0,2000000.0,4.0,0,1,709.0
006-002-0023879-0,2922,1902,2860501.0,176000000.0,113842595.0,1008570,3023534,2,320385440,54841.0,...,educacion_basica_secundaria,1,antioquia,0,22020000.0,1500000.0,4.0,1,0,733.0
006-002-0026159-4,2557,1902,987637.0,50300000.0,38521256.0,317167,1023082,2,320385440,54841.0,...,educacion_basica_secundaria,1,antioquia,0,22020000.0,1500000.0,4.0,1,0,695.0


In [245]:
from sklearn.model_selection import train_test_split

X = df.drop("default", axis=1)
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=1
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

Training set size: (10288, 21)
Testing set size: (2572, 21)


In [246]:
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

model = LGBMClassifier(
    objective="binary",
    class_weight="balanced",
    verbose=0,
    random_state=1,
)

train_x, val_x, train_y, val_y = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
)

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

model.fit(
    train_x,
    train_y,
    categorical_feature=cat_vars,
    eval_set=[(val_x, val_y)], 
    eval_metric="logloss",
    callbacks=[lgb.early_stopping(stopping_rounds=20)],
)

train_score = f1_score(y_train, model.predict(X_train))
test_score = f1_score(y_test, model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's binary_logloss: 0.21411
Train score: 0.90
Test score: 0.77


In [247]:
from sklearn.metrics import classification_report, confusion_matrix


def print_resultados(model, X_test, y_test):
    print('reporte de clasificación:')
    print(classification_report(y_test, model.predict(X_test)), '\n')
    print('matriz de confusión:')
    print(confusion_matrix(y_test, model.predict(X_test)), '\n')
    feature_importances = pd.Series(model.feature_importances_, index=X_train.columns)
    feature_importances.sort_values(ascending=False, inplace=True)
    print('10 características más importantes:')
    print(feature_importances.head(10))
    return

In [248]:
print_resultados(model, X_test, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.95      0.95      2126
           1       0.75      0.79      0.77       446

    accuracy                           0.92      2572
   macro avg       0.85      0.87      0.86      2572
weighted avg       0.92      0.92      0.92      2572
 

matriz de confusión:
[[2010  116]
 [  94  352]] 

10 características más importantes:
s_intereses         445
vinculacion         360
puntaje_data        255
ctasahorros         227
v_cuota             222
valorgarantia       220
curtotalingresos    205
s_capital           183
plazo               155
v_prestamo          135
dtype: int32


In [249]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from xgboost import XGBClassifier

te = TargetEncoder()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

train_x, val_x, train_y, val_y = train_test_split(
    X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
)

model = XGBClassifier(
    grow_policy="lossguide",
    tree_method="hist",
    early_stopping_rounds=20,
    eval_metric="auc",
    random_state=1,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()
)

model.fit(
    train_x, 
    train_y, 
    eval_set=[(val_x, val_y)], 
    verbose=False
    )

train_score = f1_score(y_train, model.predict(X_train_processed), average="macro")
test_score = f1_score(y_test, model.predict(X_test_processed), average="macro")
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Train score: 0.94
Test score: 0.85


In [250]:
print_resultados(model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.95      0.94      0.95      2126
           1       0.74      0.78      0.76       446

    accuracy                           0.91      2572
   macro avg       0.85      0.86      0.85      2572
weighted avg       0.92      0.91      0.91      2572
 

matriz de confusión:
[[2004  122]
 [ 100  346]] 

10 características más importantes:
actualizacion       0.473580
estado_cliente      0.080540
tipoasociado        0.055923
garantias           0.042053
edad                0.040274
puntaje_data        0.035806
curtotalingresos    0.031459
s_intereses         0.030279
v_prestamo          0.025728
s_capital           0.020757
dtype: float32


In [251]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=1, max_depth=5)
model.fit(X_train_processed, y_train)
train_score = f1_score(y_train, model.predict(X_train_processed))
test_score = f1_score(y_test, model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Train score: 0.71
Test score: 0.65


In [252]:
print_resultados(model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.92      0.95      0.93      2126
           1       0.70      0.60      0.65       446

    accuracy                           0.89      2572
   macro avg       0.81      0.77      0.79      2572
weighted avg       0.88      0.89      0.88      2572
 

matriz de confusión:
[[2012  114]
 [ 178  268]] 

10 características más importantes:
actualizacion         0.274756
edad                  0.267735
garantias             0.174670
tipoasociado          0.074015
s_intereses           0.070620
v_prestamo            0.066837
curtotalingresos      0.058769
actividadeconomica    0.004937
estado_cliente        0.003663
vinculacion           0.002496
dtype: float64


# Sintonización de modelos

### LightGBM

In [253]:
import lightgbm as lgb
import optuna


def objective(trial):
    param = {
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 50, 500),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    categorical_features = X_train.select_dtypes(include=["category"]).columns.tolist()

    train_x, val_x, train_y, val_y = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = LGBMClassifier(
        **param,
        objective="binary",
        force_col_wise=True,
        random_state=1,
        verbose=-1,
    )

    model.fit(
        train_x,
        train_y,
        categorical_feature=categorical_features,
        eval_set=[(val_x, val_y)],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

lgb_model = LGBMClassifier(
    **study.best_trial.params,
    objective="binary",
    force_col_wise=True,
    random_state=1,
)

lgb_model.fit(X_train, y_train)
lgb_params = lgb_model.get_params()

train_score = f1_score(y_train, lgb_model.predict(X_train))
test_score = f1_score(y_test, lgb_model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")


Best trial: 0.778 with params: {'lambda_l1': 0.1235393118590831, 'lambda_l2': 0.0008438947206701026, 'num_leaves': 125, 'feature_fraction': 0.4659683204413405, 'bagging_fraction': 0.48117393721243357, 'bagging_freq': 6, 'min_child_samples': 39, 'learning_rate': 0.04156752650085056, 'colsample_bytree': 0.900798987510449, 'scale_pos_weight': 4.485504915153962}
Train score: 0.88
Test score: 0.78


In [254]:
print_resultados(lgb_model, X_test, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      2126
           1       0.74      0.82      0.78       446

    accuracy                           0.92      2572
   macro avg       0.85      0.88      0.86      2572
weighted avg       0.92      0.92      0.92      2572
 

matriz de confusión:
[[2000  126]
 [  82  364]] 

10 características más importantes:
s_intereses         986
valorgarantia       760
vinculacion         709
curtotalingresos    658
ctasahorros         622
puntaje_data        586
aportes             558
v_cuota             545
s_capital           544
edad                435
dtype: int32


### XGBoost

In [255]:
def objective(trial):

    te = TargetEncoder()

    cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("target_encoder", te, cat_vars),
        ],
        remainder="passthrough",
    )

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_test_processed = preprocessor.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 5e-1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1e2, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 1e2, log=True),
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
        "gamma": trial.suggest_float("gamma", 0, 2),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 10),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10)
    }

    model = XGBClassifier(
        grow_policy="lossguide",
        tree_method="hist",
        early_stopping_rounds=20,
        eval_metric="auc",
        random_state=1,
       )

    model.fit(
        train_x, 
        train_y, 
        eval_set=[(val_x, val_y)], 
        verbose=False
        )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

xgb_model = XGBClassifier(
    **study.best_trial.params,
    grow_policy="lossguide",
    tree_method="hist",
    #early_stopping_rounds=20,
    #eval_metric="auc",
    random_state=1,
)

te = TargetEncoder()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

xgb_model.fit(X_train_processed, y_train)
xgb_params = xgb_model.get_params()

train_score = f1_score(y_train, xgb_model.predict(X_train_processed))
test_score = f1_score(y_test, xgb_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Best trial: 0.783 with params: {'max_depth': 8, 'learning_rate': 0.2425175183526999, 'n_estimators': 335, 'subsample': 0.6260938383179238, 'colsample_bytree': 0.6438343535068245, 'reg_alpha': 0.4395897770525273, 'reg_lambda': 1.0554307768386781, 'max_delta_step': 1, 'gamma': 1.645005120769016, 'min_child_weight': 8, 'scale_pos_weight': 3.5568903291346095}
Train score: 0.98
Test score: 0.78


In [256]:
print_resultados(xgb_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.95      0.96      0.95      2126
           1       0.80      0.76      0.78       446

    accuracy                           0.92      2572
   macro avg       0.87      0.86      0.87      2572
weighted avg       0.92      0.92      0.92      2572
 

matriz de confusión:
[[2039   87]
 [ 108  338]] 

10 características más importantes:
actualizacion     0.502964
estado_cliente    0.070432
tipoasociado      0.042221
garantias         0.037762
edad              0.034027
ctasahorros       0.031972
valorgarantia     0.030862
v_prestamo        0.027468
puntaje_data      0.026775
s_capital         0.023242
dtype: float32


### Random Forest

In [257]:
from sklearn.ensemble import RandomForestClassifier


def objective(trial):

    te = TargetEncoder()

    cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("target_encoder", te, cat_vars),
        ],
        remainder="passthrough",
    )

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_test_processed = preprocessor.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = RandomForestClassifier(
        class_weight="balanced",
        random_state=1
        )

    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical("max_features", 
        ["sqrt", "log2", None]),
    }
      
    model.fit(
        train_x, 
        train_y, 
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

rf_model = RandomForestClassifier(
    **study.best_trial.params,
    class_weight='balanced',
    random_state=1,
)

te = TargetEncoder()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

rf_model.fit(X_train_processed, y_train)
rf_params = rf_model.get_params()

train_score = f1_score(y_train, rf_model.predict(X_train_processed))
test_score = f1_score(y_test, rf_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Best trial: 0.734 with params: {'n_estimators': 56, 'max_depth': 4, 'min_samples_split': 8, 'min_samples_leaf': 17, 'max_features': 'log2'}
Train score: 0.68
Test score: 0.67


In [258]:
print_resultados(rf_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.86      0.91      2126
           1       0.56      0.84      0.67       446

    accuracy                           0.86      2572
   macro avg       0.76      0.85      0.79      2572
weighted avg       0.89      0.86      0.87      2572
 

matriz de confusión:
[[1833  293]
 [  70  376]] 

10 características más importantes:
actualizacion       0.267851
tipoasociado        0.167350
garantias           0.140228
curtotalingresos    0.090998
puntaje_data        0.066462
edad                0.064613
v_prestamo          0.055937
estado_cliente      0.053257
valorgarantia       0.039038
s_capital           0.017513
dtype: float64


### Logistic Regression

In [259]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PowerTransformer


def objective(trial):

    te = TargetEncoder()
    pt = PowerTransformer()

    cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()
    num_vars = ['plazo', 'v_cuota', 'v_prestamo', 's_capital',
       's_intereses', 'aportes', 'valorgarantia', 'ctasahorros',
       'edad', 'curtotalingresos', 'curtotalegresos',
       'puntaje_data']

    preprocessor = ColumnTransformer(
        transformers=[
            ("target_encoder", te, cat_vars),
            ("power_transformer", pt, num_vars),
        ],
        remainder="passthrough",
    )

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_test_processed = preprocessor.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = LogisticRegression(
        class_weight="balanced",
        solver="saga",
        max_iter=10000,
        random_state=1
        )

    param = {
        "C": trial.suggest_float("C", 1e-3, 1e3, log=True),
        #"l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "penalty": trial.suggest_categorical("penalty", ["l1", "l2", None]),
    }
      
    model.fit(
        train_x, 
        train_y, 
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

lr_model = LogisticRegression(
    **study.best_trial.params,
    class_weight='balanced',
    solver="saga",
    max_iter=10000,
    random_state=1,
)

te = TargetEncoder()
pt = PowerTransformer()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

num_vars = ['plazo', 'v_cuota', 'v_prestamo', 's_capital',
       's_intereses', 'aportes', 'valorgarantia', 'ctasahorros',
       'edad', 'curtotalingresos', 'curtotalegresos',
       'puntaje_data']

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
        ("power_transformer", pt, num_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

lr_model.fit(X_train_processed, y_train)
lr_params = lr_model.get_params()

train_score = f1_score(y_train, lr_model.predict(X_train_processed))
test_score = f1_score(y_test, lr_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Best trial: 0.472 with params: {'C': 0.0035702889847891326, 'penalty': None}
Train score: 0.49
Test score: 0.48


In [260]:
print(classification_report(lr_model.predict(X_test_processed), y_test))
print(confusion_matrix(lr_model.predict(X_test_processed),y_test))

              precision    recall  f1-score   support

           0       0.70      0.94      0.80      1583
           1       0.77      0.35      0.48       989

    accuracy                           0.71      2572
   macro avg       0.74      0.64      0.64      2572
weighted avg       0.73      0.71      0.68      2572

[[1482  101]
 [ 644  345]]


In [261]:
pesos = pd.Series(lr_model.coef_[0], index=X_train.columns.to_list())
pesos.loc['intercept'] = lr_model.intercept_[0]
pesos.sort_values(ascending=False)

aportes               0.052554
curtotalingresos      0.026823
v_cuota               0.008246
curtotalegresos       0.003599
actualizacion         0.002455
intercept             0.001897
plazo                 0.001338
vinculacion           0.000446
departamento         -0.000010
s_intereses          -0.001940
intestrato           -0.007931
s_capital            -0.014232
edad                 -0.016995
actividadeconomica   -0.021977
v_prestamo           -0.024582
tipoasociado         -0.026534
sexo                 -0.038085
puntaje_data         -0.042335
garantias            -0.043891
valorgarantia        -0.045482
estado_cliente       -0.052592
ctasahorros          -0.079402
dtype: float64

### Multilayer Perceptron

In [262]:
from sklearn.neural_network import MLPClassifier


def objective(trial):

    te = TargetEncoder()
    pt = PowerTransformer()

    cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()
    num_vars = ['plazo', 'v_cuota', 'v_prestamo', 's_capital',
       's_intereses', 'aportes', 'valorgarantia', 'ctasahorros',
       'edad', 'curtotalingresos', 'curtotalegresos',
       'puntaje_data']

    preprocessor = ColumnTransformer(
        transformers=[
            ("target_encoder", te, cat_vars),
            ("power_transformer", pt, num_vars),
        ],
        remainder="passthrough",
    )

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_test_processed = preprocessor.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = MLPClassifier(
        max_iter=1000,
    )

    param = {
        "alpha": trial.suggest_float("alpha", 1e-3, 1e3, log=True),
        "learning_rate_init": trial.suggest_float("learning_rate_init", 0.001, 0.1, log=True),
        "hidden_layer_sizes": trial.suggest_categorical("hidden_layer_sizes", [
            (10, 5),
            (20, 10),
            (20,),
            (50, 20),
            (100, 20),
            (100,),
        ]),
        "activation": trial.suggest_categorical("activation", [
            "logistic", "tanh", "relu"]),
    }
      
    model.fit(
        train_x, 
        train_y, 
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_nn_model = MLPClassifier(
    **study.best_trial.params,
    max_iter=1000,
    )

te = TargetEncoder()
pt = PowerTransformer()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

num_vars = ['plazo', 'v_cuota', 'v_prestamo', 's_capital',
       's_intereses', 'aportes', 'valorgarantia', 'ctasahorros',
       'edad', 'curtotalingresos', 'curtotalegresos',
       'puntaje_data']

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
        ("power_transformer", pt, num_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

sk_nn_model.fit(X_train_processed, y_train)
sk_nn_params = sk_nn_model.get_params()

train_score = f1_score(y_train, sk_nn_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_nn_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Best trial: 0.684 with params: {'alpha': 18.358423930429836, 'learning_rate_init': 0.0024642526643145857, 'hidden_layer_sizes': (100,), 'activation': 'tanh'}
Train score: 0.00
Test score: 0.00


# Modelos con Skrub

### LightGBM

In [263]:
import warnings

from skrub import TableVectorizer

warnings.filterwarnings("ignore")

def objective(trial):
    param = {
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 50, 500),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }


    vectorizer = TableVectorizer()

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
    X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = LGBMClassifier(
        **param,
        objective="binary",
        force_col_wise=True,
        random_state=1,
        verbose=-1,
    )

    model.fit(
        train_x,
        train_y,
        eval_set=[(val_x, val_y)],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_lgb_model = LGBMClassifier(
    **study.best_trial.params,
    objective="binary",
    force_col_wise=True,
    random_state=1,
)

sk_lgb_model.fit(X_train_processed, y_train)
sk_lgb_params = sk_lgb_model.get_params()

train_score = f1_score(y_train, sk_lgb_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_lgb_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Best trial: 0.783 with params: {'lambda_l1': 0.0064652346184465356, 'lambda_l2': 2.6568111342419385e-06, 'num_leaves': 438, 'feature_fraction': 0.7280621833047148, 'bagging_fraction': 0.5112026705527561, 'bagging_freq': 6, 'min_child_samples': 31, 'learning_rate': 0.09873042518215987, 'colsample_bytree': 0.8795857488623939, 'scale_pos_weight': 4.600060913631212}
Train score: 0.99
Test score: 0.76


In [264]:
def print_resultados_skrub(model, X_test, y_test):
    print('reporte de clasificación:')
    print(classification_report(y_test, model.predict(X_test)), '\n')
    print('matriz de confusión:')
    print(confusion_matrix(y_test, model.predict(X_test)), '\n')
    feature_importances = pd.Series(
        model.feature_importances_, 
        index=model.feature_names_in_)
    feature_importances.sort_values(ascending=False, inplace=True)
    print('10 características más importantes:')
    print(feature_importances.head(10))
    return

In [265]:
print_resultados_skrub(sk_lgb_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.95      0.96      0.95      2126
           1       0.79      0.74      0.76       446

    accuracy                           0.92      2572
   macro avg       0.87      0.85      0.86      2572
weighted avg       0.92      0.92      0.92      2572
 

matriz de confusión:
[[2036   90]
 [ 117  329]] 

10 características más importantes:
Column_6     1216
Column_13    1100
Column_9     1097
Column_11     981
Column_14     977
Column_8      896
Column_5      856
Column_3      758
Column_7      733
Column_10     683
dtype: int32


### Skrub con modelo por defecto

In [266]:
from skrub import tabular_pipeline


def objective(trial):
    
    
    param = {
        "histgradientboostingclassifier__learning_rate": trial.suggest_float("histgradientboostingclassifier__learning_rate", 0.01, 0.2),  # noqa: E501
        "histgradientboostingclassifier__max_iter": trial.suggest_int("histgradientboostingclassifier__max_iter", 100, 500),  # noqa: E501
        "histgradientboostingclassifier__max_leaf_nodes": trial.suggest_int("histgradientboostingclassifier__max_leaf_nodes", 20, 100), # noqa: E501
        "histgradientboostingclassifier__min_samples_leaf": trial.suggest_int("histgradientboostingclassifier__min_samples_leaf", 1, 10), # noqa: E501
        "histgradientboostingclassifier__l2_regularization": trial.suggest_float("histgradientboostingclassifier__l2_regularization", 1e-3, 10.0, log=True), # noqa: E501
        "histgradientboostingclassifier__early_stopping": trial.suggest_categorical("histgradientboostingclassifier__early_stopping", [True, False]), # noqa: E501
    }

    train_x, val_x, train_y, val_y = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = tabular_pipeline(estimator='classifier')

    model.fit(
        train_x,
        train_y,
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_params = study.best_trial.params
sk_params = {k.replace('histgradientboostingclassifier__', ''): v for k, v in sk_params.items()}  # noqa: E501
sk_model = tabular_pipeline(estimator='classifier')
sk_model["histgradientboostingclassifier"].set_params(**sk_params)
sk_model.fit(X_train, y_train)
sk_params = sk_model.get_params()

train_score = f1_score(y_train, sk_model.predict(X_train))
test_score = f1_score(y_test, sk_model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Best trial: 0.775 with params: {'histgradientboostingclassifier__learning_rate': 0.09669079422253268, 'histgradientboostingclassifier__max_iter': 411, 'histgradientboostingclassifier__max_leaf_nodes': 35, 'histgradientboostingclassifier__min_samples_leaf': 3, 'histgradientboostingclassifier__l2_regularization': 0.0014794174369746611, 'histgradientboostingclassifier__early_stopping': True}
Train score: 0.95
Test score: 0.77


In [267]:
print('reporte de clasificación:')
print(classification_report(y_test, sk_model.predict(X_test)), '\n')
print('matriz de confusión:')
print(confusion_matrix(y_test, sk_model.predict(X_test)), '\n')

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.94      0.97      0.96      2126
           1       0.84      0.70      0.77       446

    accuracy                           0.93      2572
   macro avg       0.89      0.84      0.86      2572
weighted avg       0.92      0.93      0.92      2572
 

matriz de confusión:
[[2068   58]
 [ 132  314]] 



### XGBoost

In [268]:
def objective(trial):

    vectorizer = TableVectorizer()

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 5e-1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1e2, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 1e2, log=True),
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
        "gamma": trial.suggest_float("gamma", 0, 2),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 10),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10)
    }

    model = XGBClassifier(
        grow_policy="lossguide",
        tree_method="hist",
        early_stopping_rounds=20,
        eval_metric="auc",
        random_state=1,
       )

    model.fit(
        train_x, 
        train_y, 
        eval_set=[(val_x, val_y)], 
        verbose=False
        )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_xgb_model = XGBClassifier(
    **study.best_trial.params,
    grow_policy="lossguide",
    tree_method="hist",
    random_state=1,
)

vectorizer = TableVectorizer()

X_train_processed = vectorizer.fit_transform(X_train, y_train)
X_test_processed = vectorizer.transform(X_test)

sk_xgb_model.fit(X_train_processed, y_train)
sk_xgb_params = xgb_model.get_params()

train_score = f1_score(y_train, sk_xgb_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_xgb_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Best trial: 0.785 with params: {'max_depth': 8, 'learning_rate': 0.08471387013040654, 'n_estimators': 119, 'subsample': 0.5870216321662268, 'colsample_bytree': 0.7623661853988232, 'reg_alpha': 0.44006206983507024, 'reg_lambda': 0.002110375764685296, 'max_delta_step': 0, 'gamma': 1.0804831692936636, 'min_child_weight': 8, 'scale_pos_weight': 3.2218352122981315}
Train score: 0.93
Test score: 0.78


In [269]:
print_resultados_skrub(sk_xgb_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.95      0.95      2126
           1       0.77      0.79      0.78       446

    accuracy                           0.92      2572
   macro avg       0.86      0.87      0.87      2572
weighted avg       0.92      0.92      0.92      2572
 

matriz de confusión:
[[2023  103]
 [  95  351]] 

10 características más importantes:
actualizacion            0.309874
tipoasociado             0.038483
ctasahorros              0.032212
actividadeconomica_16    0.028316
garantias                0.028290
valorgarantia            0.027723
s_intereses              0.024994
vinculacion              0.019740
puntaje_data             0.019551
actividadeconomica_15    0.019163
dtype: float32


### Random Forest

In [271]:
from sklearn.ensemble import RandomForestClassifier


def objective(trial):


    vectorizer = TableVectorizer()

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)
    
    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = RandomForestClassifier(
        class_weight="balanced",
        random_state=1
        )

    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical("max_features", 
        ["sqrt", "log2", None]),
    }
      
    model.fit(
        train_x, 
        train_y, 
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_rf_model = RandomForestClassifier(
    **study.best_trial.params,
    class_weight='balanced',
    random_state=1,
)

vectorizer = TableVectorizer()

X_train_processed = vectorizer.fit_transform(X_train, y_train)
X_test_processed = vectorizer.transform(X_test)

sk_rf_model.fit(X_train_processed, y_train)
sk_rf_params = sk_rf_model.get_params()

train_score = f1_score(y_train, sk_rf_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_rf_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Best trial: 0.714 with params: {'n_estimators': 86, 'max_depth': 7, 'min_samples_split': 19, 'min_samples_leaf': 5, 'max_features': 'log2'}
Train score: 0.74
Test score: 0.71


In [272]:
print_resultados_skrub(sk_rf_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.90      0.93      2126
           1       0.63      0.82      0.71       446

    accuracy                           0.88      2572
   macro avg       0.79      0.86      0.82      2572
weighted avg       0.90      0.88      0.89      2572
 

matriz de confusión:
[[1909  217]
 [  79  367]] 

10 características más importantes:
actualizacion       0.204947
ctasahorros         0.135309
s_intereses         0.121294
curtotalingresos    0.100199
puntaje_data        0.065908
valorgarantia       0.057727
vinculacion         0.054783
tipoasociado        0.050072
aportes             0.047824
v_cuota             0.025798
dtype: float64


### Logistic regression

In [273]:
def objective(trial):

    vectorizer = TableVectorizer()
 #   te = TargetEncoder()
    pt = PowerTransformer()

    cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()
    num_vars = ['plazo', 'v_cuota', 'v_prestamo', 's_capital',
       's_intereses', 'aportes', 'valorgarantia', 'ctasahorros',
       'edad', 'curtotalingresos', 'curtotalegresos',
       'puntaje_data']

    preprocessor = ColumnTransformer(
        transformers=[
#            ("target_encoder", te, cat_vars),
            ("power_transformer", pt, num_vars),
            ("vectorizer", vectorizer, cat_vars),
        ],
        remainder="passthrough",
    )

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_test_processed = preprocessor.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = LogisticRegression(
        class_weight="balanced",
        solver="saga",
        max_iter=10000,
        random_state=1
        )

    param = {
        "C": trial.suggest_float("C", 1e-3, 1e3, log=True),
        #"l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "penalty": trial.suggest_categorical("penalty", ["l1", "l2", None]),
    }
      
    model.fit(
        train_x, 
        train_y, 
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_lr_model = LogisticRegression(
    **study.best_trial.params,
    class_weight='balanced',
    solver="saga",
    max_iter=10000,
    random_state=1,
)

vectorizer = TableVectorizer()
pt = PowerTransformer()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

num_vars = ['plazo', 'v_cuota', 'v_prestamo', 's_capital',
       's_intereses', 'aportes', 'valorgarantia', 'ctasahorros',
       'edad', 'curtotalingresos', 'curtotalegresos',
       'puntaje_data']

preprocessor = ColumnTransformer(
    transformers=[
        ("vectorizer", vectorizer, cat_vars),
        ("power_transformer", pt, num_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

sk_lr_model.fit(X_train_processed, y_train)
sk_lr_params = sk_lr_model.get_params()

train_score = f1_score(y_train, sk_lr_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_lr_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Best trial: 0.469 with params: {'C': 0.5126577392757116, 'penalty': 'l2'}
Train score: 0.49
Test score: 0.48


In [274]:
print(classification_report(sk_lr_model.predict(X_test_processed), y_test))
print(confusion_matrix(sk_lr_model.predict(X_test_processed),y_test))

              precision    recall  f1-score   support

           0       0.69      0.94      0.79      1557
           1       0.78      0.34      0.48      1015

    accuracy                           0.70      2572
   macro avg       0.73      0.64      0.64      2572
weighted avg       0.72      0.70      0.67      2572

[[1460   97]
 [ 666  349]]


In [275]:
pesos = pd.Series(sk_lr_model.coef_[0], index=preprocessor.get_feature_names_out())
pesos.loc['intercept'] = sk_lr_model.intercept_[0]
pesos.sort_values(ascending=False)

power_transformer__s_intereses       0.052489
remainder__tipoasociado              0.026682
power_transformer__plazo             0.008300
vectorizer__actividadeconomica_00    0.003987
remainder__estado_cliente            0.003347
                                       ...   
remainder__actualizacion            -0.042411
power_transformer__aportes          -0.043681
power_transformer__valorgarantia    -0.045273
power_transformer__puntaje_data     -0.052475
power_transformer__ctasahorros      -0.079287
Length: 73, dtype: float64

### Multilayer Perceptron

In [276]:
from sklearn.neural_network import MLPClassifier


def objective(trial):

    vectorizer = TableVectorizer()
    pt = PowerTransformer()

    cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()
    num_vars = ['plazo', 'v_cuota', 'v_prestamo', 's_capital',
       's_intereses', 'aportes', 'valorgarantia', 'ctasahorros',
       'edad', 'curtotalingresos', 'curtotalegresos',
       'puntaje_data']

    preprocessor = ColumnTransformer(
        transformers=[
#            ("target_encoder", te, cat_vars),
            ("power_transformer", pt, num_vars),
            ("vectorizer", vectorizer, cat_vars),
        ],
        remainder="passthrough",
    )

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_test_processed = preprocessor.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = MLPClassifier(
        max_iter=1000,
    )

    param = {
        "alpha": trial.suggest_float("alpha", 1e-3, 1e3, log=True),
        "learning_rate_init": trial.suggest_float("learning_rate_init", 0.001, 0.1, log=True),
        "hidden_layer_sizes": trial.suggest_categorical("hidden_layer_sizes", [
            (10, 5),
            (20, 10),
            (20,),
            (50, 20),
            (100, 20),
            (100,),
        ]),
        "activation": trial.suggest_categorical("activation", [
            "logistic", "tanh", "relu"]),
    }
      
    model.fit(
        train_x, 
        train_y, 
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_nn_model = MLPClassifier(
    **study.best_trial.params,
    max_iter=1000,
    )

vectorizer = TableVectorizer()
pt = PowerTransformer()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

num_vars = ['plazo', 'v_cuota', 'v_prestamo', 's_capital',
       's_intereses', 'aportes', 'valorgarantia', 'ctasahorros',
       'edad', 'curtotalingresos', 'curtotalegresos',
       'puntaje_data']

preprocessor = ColumnTransformer(
    transformers=[
        ("vectorizer", vectorizer, cat_vars),
        ("power_transformer", pt, num_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

sk_nn_model.fit(X_train_processed, y_train)
sk_nn_params = sk_nn_model.get_params()

train_score = f1_score(y_train, sk_nn_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_nn_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Best trial: 0.702 with params: {'alpha': 0.10312793997715194, 'learning_rate_init': 0.034512260220018146, 'hidden_layer_sizes': (100,), 'activation': 'tanh'}
Train score: 0.00
Test score: 0.00
